In [1]:
import numpy as np
from matplotlib import pyplot as plt
import scipy.stats as stats
from pycircstat2.descriptive import circ_mean_and_r


plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.spines.top"] = False
plt.rcParams["font.size"] = 12
plt.rcParams["legend.frameon"] = False

In [ ]:
lr = 0.01
a = 10
N = 500
N_i = 500
prop_shift = 0.0
hebb_k = 0
eta_k = 1.0
w_clip = True
n_days = 30
vars_ef = np.random.lognormal(mean=2, sigma=0.6, size=N)

def gaussian(x, mu, sigma):
    return np.exp(-(x - mu)**2/(2 * sigma**2)) * 1/(np.sqrt(2 * np.pi) * sigma)

def init_weights(N, N_i):

    vars_if = np.random.lognormal(mean=3, sigma=0.6, size=N_i)
    varss_ei = np.random.lognormal(mean=3, sigma=0.6, size=N)

    x = np.linspace(0, 180, N)
    matrix = np.zeros((N, N))
    for i in range(N):
        matrix[:, i] = stats.norm.pdf(x, x[i], vars_ef[i]) + stats.norm.pdf(x, x[i] + 180, vars_ef[i]) + stats.norm.pdf(x, x[i] - 180, vars_ef[i])
    w = matrix/N
    w /= np.sum(w, axis=0)

    x = np.linspace(0, 180, N_i)
    matrix = np.zeros((N, N_i))
    for i in range(N_i):
        matrix[:, i] = stats.norm.pdf(x, x[i], vars_if[i]) + stats.norm.pdf(x, x[i] + 180, vars_if[i]) + stats.norm.pdf(x, x[i] - 180, vars_if[i])
    w_if = matrix/N
    w_if /= np.sum(w_if, axis=0)

    x = np.linspace(0, 180, N)
    matrix = np.zeros((N_i, N))
    for i in range(N):
        matrix[:, i] = stats.norm.pdf(x, x[i], varss_ei[i]) + stats.norm.pdf(x, x[i] + 180, varss_ei[i]) + stats.norm.pdf(x, x[i] - 180, varss_ei[i])
    w_ei = matrix/N_i
    w_ei /= np.sum(w_ei, axis=0)

    return w, w_if, w_ei
    

def circular_gaussian(theta, amp=0.62, sigma=25, baseline=0):
    theta_y = np.linspace(0, 180, N)
    d = np.abs(theta - theta_y)
    d_plus = d + 180
    d_minus = d - 180
    y = amp * (np.exp(-d**2/(2 * sigma**2)) + np.exp(-d_plus**2/(2 * sigma**2)) + np.exp(-d_minus**2/(2 * sigma**2))) + baseline
    return y

def noise(w):
    return np.random.randn(*w.shape)  # random noise to simulate variability in plasticity

def normalize(x):
    return x / np.sum(x, axis=0, keepdims=True)  # normalize weights to sum to 1

def propensity(x):
    return np.tanh((a * x) + prop_shift)  # example propensity function

def circular_dist(x, y):
    return np.minimum(np.abs(x - y), 180 - np.abs(x - y))  # circular distance between two angles


def step(w, w_if, w_ei, I_cno=0):
    theta = np.random.uniform(0, 180)  # random phase
    r_f =  circular_gaussian(theta)# activity of the stimulus locked presynaptic population
    r_i =  np.clip(w_if.T @ r_f - I_cno, 0, np.inf)# activity of inhibitory population
    r_E =  np.clip(w.T @ r_f - w_ei.T @ r_i, 0, np.inf)  # activity of the single excitatory neuron of interest

    return r_f, r_i, r_E

def plasticity_rule(r_f, r_i, r_E, w, w_max=None):
    # Hebbian term
    hebbian_term = np.outer(r_f, r_E)

    # Update weights based on the plasticity rule
    dw = lr * propensity(w) * ((hebb_k * hebbian_term) - (eta_k*noise(w)))
    w_new = w + dw
    if w_clip:
        w_new = np.clip(w_new, 0, w_max)  # clip weights to be non-negative
    return w_new

def PO_estimate(w, w_if, w_ei, I_cno=0, method='argmax', min_r=0.05):
    n_theta = 100
    theta_list = np.linspace(0, 180, n_theta, endpoint=False)
    activity = np.zeros((N, n_theta))

    for theta_idx, theta in enumerate(theta_list):
        r_f = circular_gaussian(theta)
        r_i = np.clip(w_if.T @ r_f - I_cno, 0, np.inf)
        r_E = np.clip(w.T @ r_f - w_ei.T @ r_i, 0, np.inf)
        activity[:, theta_idx] = r_E

    if method == 'argmax':
        peak = activity.max(axis=1)
        return np.where(peak > 0, theta_list[np.argmax(activity, axis=1)], np.nan)  # return NaN if peak is below threshold
        
    elif method == 'circular_mean':
        alpha2 = 2 * np.deg2rad(theta_list)
        POs = np.full(N, np.nan)  # Initialize POs with NaN
        for i in range(N):
            if activity[i].sum() <= 1e-12: # silent: skip, avoids nan warning
                continue
            mu, r = circ_mean_and_r(alpha2, w=activity[i])
            if np.isnan(mu) or r < min_r: # untuned: no meaningful PO
                continue
            POs[i] = (np.rad2deg(mu / 2)) % 180  # halve back to [0, 180)
        return POs
    else:
        raise ValueError("Invalid method")
    
    return POs

def run_sim():

    n_steps = 300 + n_days * 30
    n_steps_per_norm = 30

    r_f_rec = []
    r_i_rec = []
    r_e_rec = []

    POs = []
    w,w_if, w_ei = init_weights(N, N_i)
    w_rec = [w]
    if w_clip:
        w_max = 1.5 * np.percentile(w, 99)  # set w_max to the 99th percentile of initial weights
    for t in range(n_steps):
        r_f, r_i, r_E = step(w, w_if, w_ei)
        w = plasticity_rule(r_f, r_i, r_E, w, w_max=w_max if w_clip else None)

        # r_f_rec.append(r_f); r_i_rec.append(r_i); r_e_rec.append(r_E)


        if (t + 1) % n_steps_per_norm == 0:
            w = normalize(w)  # normalize weights after each update
        w_rec.append(w)
        POs.append(PO_estimate(w, w_if, w_ei, method='circular_mean', min_r=0.05))

    return r_f_rec, r_i_rec, r_e_rec, w_rec, POs

def drift_metrics(POs, ref_angle=90):
    preferences = POs.T
    initial_preferences = preferences[:, 0]

    drift_mag = np.array([circular_dist(initial_preferences, preferences[:, t]) for t in range(preferences.shape[1])])
    drift_rate = np.array([circular_dist(preferences[:, t], preferences[:, t + 1]) for t in range(preferences.shape[1] - 1)])

    initial_dist = circular_dist(initial_preferences, ref_angle)
    distances = circular_dist(preferences, ref_angle)
    convergence = np.array([initial_dist - distances[:, t] for t in range(preferences.shape[1] - 1)])
    return drift_mag, drift_rate, convergence